here  we will ad the possitional incoding part of thealgorithm , so  from begginning to feed Forward class definition code will be the same as gpt2.ipynb  then some utility functions will be add and after that the  positional encodding class define. then in the attention class the possitional class is instatiatet  as a self attribute  and then the rotations are apply to queries and key matrixes  in the forward function. then we downdload from Hugging face the weigths from a gpt2 trainned using Rope  

In [ ]:
!pip install xformers einops np

In [ ]:
import numpy as np
import torch
import torch.nn.functional as functional
from torch.nn.modules import ModuleList
from toch.nn.modules.normalization import LayerNorm
from torch import nn,einsum,broadcast_tensors
from einops import rearrange,repeat
import copy
import math

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
def _get_clones(module,n):
    return ModuleList([copy.deepcopy(module) for i in range (n)])
class Conv1D (nn.Module):
    def __init__(self,input,output):
        super().__init__()
        self.output=output
        W=torch.empty(input,output) 
        #because the matrix is inXout there is no need to transpose in forward
        nn.init.normal(W,std=0.2)
        self.weight=nn.Parameter(W)
        self.bias=nn.Parameter(torch.zeros(output))
    def forward(self,x):
        """x= input tensor"""
        size_out=x.size()[:-1]+(self.output,)
        x=torch.addmm(self.bias,x.view(-1,x.size(-1),self.weight))
        x=x.view(*size_out)
        return x
        
        
        
class FeedForward(nn.Module):
    def __init__(self,dropout,d_model=768,d_hide=768*4):
        super().__init__()
        self.l1=Conv1D(d_model,d_hide)
        self.activ=functional.gelu
        self.l2=Conv1D(d_hide,d_model)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        return self.dropout(self.l2(self.activ(self.l1(x))))

Utility functions for Rope

In [ ]:
def exists (val):
    return val is not None
def default (val,d):
    return val if exists (val) else d
def broadCat (tensors,dim=-1):
    broadcasted_tensors=broadcast_tensors(*tensors)
    return torch.cat(broadcasted_tensors,dim=dim)
def rotate_half(x):
    """
    The initial step is to split the vector in half and swap it, but with the second half negated, to account 
    for the cos, -sin part of the rotation. This is replaced with a more efficient method 
    that is allowed because of the linearity of the space, which permits dividing it into two subspaces, performing the rotation by multiplying, and then putting them back together.

    """
    x= rearrange(x,' ... (dr) -> ... d r',r=2)
    x1,x2=x.unbind(dim=-1)# the unbind  splits the tensor along the last dimention a matrix  becomes a  set of rows
    x=torch.stack(-x2,x1,dim= -1) #create a new dimto stack 
    return rearrange(x,' ... d r -> ...(dr)',r=2)
    
    
def apply_rottat_emb(freqs,t,start_idx=0,scale=1.,seq_dim=-2):
    """
    freqs: tensor of sinusoidal freqs normaly seq_len in size 
    t :tensor you want to apply this function
    start_idx: where in the last dim the the embeding should start
    seq_dim:wich axes corresponds to seq dim
    rot_dim: dimentionality of the rotation space. even
    """
    #number of rotations , the embedings size 
    rot_dim, embedding_dim=freqs.shape[-1] , t.shape[-1]
    #from the rows of freqs take the same  as the number of embedings and make sure are in the same divice as t 
    freqs=freqs[-embedding_dim:].to (t)
    end_idx=start_idx+rot_dim
    #Devides the tensor in  3 parts and does the  rotation  to the t part 
    t_left,t,t_right=t[...,:start_idx],t[...,start_idx:end_idx],t[...,end_idx]
    t = (t * freqs.cos() * scale) + (rotate_half(t) * freqs.sin() * scale)
    return torch.cat((t_left,t,t_right),dim=-1)
def learned_rotations(rotations,t,start_idx=0,freq_ranges=None):
    """ROPE  is not apply to de embedding but the K and Q matrixes asocieted with it and in the middle part of 
    those  vectores some how because rotation dont mess with dot product or norms  the semantic meannin is nor scramble"""
    #this broadcasts freq _ranges into rotations having rotation per frecbin
    if exists(freq_ranges):
        rotations=einsum('... , f -> ... f',rotations,freq_ranges )
        rotations=rearrange(rotations,'...r f -> ... (rf)')
    rotations=repeat(rotations,'... n -> ... (nr)', r=2)
    return apply_rottat_emb(rotations,t,start_idx=start_idx)
    

now the class 

In [ ]:
class RotaryEmbeddings(nn.Module):
    def __init__(
        self,
        dim,
        theta = 10000,
        max_freq = 10,
        num_freqs = 1,
        interpolate_factor = 1.,
        theta_rescale_factor = 1.,
    ):

            """    
            max_freq: max freq of rotation
            num_freq: number of times the freq is require to be iterated
            interpolate factor: a factor used to control the possitional embeding 
            theta_rescale_factor: theta value decaees so it most be re scale   
            dim : index variable that rotates from 0 to dim/2
            self.cache : are for saving precomputting rotation matrixes and not having to calculate them in the forward pass
        """
            super().__init__()
            #for longger context  sequence  he rescaling 
            theta *= theta_rescale_factor ** (dim / (dim - 2))
            #builds the frequency spectrum higther dimetions  more fine grain encoding
            freqs=1/(theta**(torch.arrange(0,dim,2)[:(dim//2)].float()/dim))
            self.cache=dict()
            self.cache_scale=dict()
            self.freqs=nn.Parameters(freqs)
            self.default_seq=-2
            assert interpolate_factor >= 1.
            self.interpolate_factor = interpolate_factor
            self.register_buffer('scale',None)
            scale = (torch.arange(0, dim, 2) + 0.4 * dim) / (1.4 * dim)
            self.register_buffer('scale', scale)

    def get_seq_pos(self,seq_len,device,dtype,offset=0):
            """
            function to obtaing  positional sequence using torch.arrange
            """
            return torch.arrange((seq_len,device=device,dtype=dtype)+offset)/self.interpolate_factor
        
    def rotate_queries_keys (self,t,seq_len=None,offset=0,freq_seq_len=None)        :
        """ Freq_seq_len is how long the enbeding thinks the seq len is , this for scaling porpuses"""
        #a helper method returns the default config--
        seq_dim=default(seq_dim,self.default_seq_dim)
        sevice, type,seq_len= t.device,t.type,t.shape[seq_dim]
        
        if exists(freq_seq_len):
            assert freq_seq_len>=seq_len
            seq_len=freq_seq_len
        freqs=self.forward(lambda:self.get_seq_posseq_len, device = device, dtype = dtype, offset = offset), cache_key = f'freqs:{seq_len}|offset:{offset}')
        ## there are implementations where seq len is a flag, not the litteral length 
        if seq_len==-3:
            freqs= rearrange(freqs,'n d -> n 1 d')
        return apply_rottat_emb(freqs, t, seq_dim = seq_dim)
    def forward(self,t,cache_key=None):
        """here we will propagate the value key """
        should_cache=exists(cache_key)
        if should_cache and cache_key in self.cache:
            self.cache[cache_key]
        if callable (t):
            t=t()
        freqs=self.freqs
        #Transposing the  frequency matrix  in just  a dimention
        freqs=einsum('...,f->...f',t.type(freqs.dtype),freqs)
        freqs=repeat(freqs,'... n -> ...(n r)',r=2)
        if should_cache:
            self.cache[cache_key]=freqs
        return freqs
        
        

now attention class where an instantiation of rotatory embedding class will be used 

In [ ]:
class Attention (nn.module):
    def init (self,d_model=768,n_heads=12,n_ctx =1024,dim_head=64,bias=True,scale=False):
        super().__init__()
        self.d_model=d_model,
        self.dim_heads=dim_head,
        self.n_heads=n_heads,
        self.scale=scale,
        self.l1=Conv1D(d_model,d_model*3),
        self.softmax=nn.Softmax(dim=1)
        self.buffer=('bias',torch.tril(torch.ones(n_ctx,n_ctx)).view(1,1,n_ctx,n_ctx))
        self.dropout=nn.Dropout(0.075)
        self.l2=Conv1D(d_model,d_model)
        self.rotate = RotaryEmbedding(dim=32)
        
    def split_heads(self,x):
        """chop of the tokens dimention and ad  a number of heads and a pice of token per head 
            dimention a the end , i will change   original code for einops code 
        """
        new_shape=x.size()[:-1]+(self.n_heads,x.size(-1)//self.n_heads)
        x=x.view(*new_shape)
        return x.permute(0,2,1,3)
    
    def _attn(self,q,k,v,attn_mask=None):
        """Remember that in attention the multiplication accours only in the last  dimention where the 
        embedion is the rest of them are  broadcasted"""
        scores=torch.matmul(q,k.transpose(-2,-1))
        if self.scale: scores=scores//math.sqrt(v.size(-1))
        q_seq_len,k_seq_len=scores.size(-2),scores.size(-1)
        if attn_mask is not None: scores=scores+attn_mask
        scores= self.softmax(scores)
        scores=self.dropout(scores)
        output=torch.matmul(scores,v)
        return outputs
            
        
        

In [ ]:
import torch.nn.functional as func  
import time
from transformers import GPT2Tokenizer
!curl --output gpt2-pytorch_model_rope.bin --location https://huggingface.co/Zuckerbird/RoPE-gpt2/resolve/main/pytorch_model.bin


In [ ]:
model=GPT2()

In [ ]:
model_dic=model.state_dic()
state_dict = torch.load("./gpt2-pytorch_model_rope.bin")
old_keys=[]
new_keys=[]
for key in state_dict.keys():
    if 'mlp' in key:
        new_key= key.replace('mlp','feedforward')
        new_keys.append(new_key)
        old_keys.append(key)

        """diferent implementations use diferent naming conventions for the feed forward block
        so it builds a maping between the loaded model and my model naming scheem"""
        

In [ ]:
for old_key,new_key in zip(old_keys,new_keys):
    state_dict[new_key]=state_dict.pop(old_key)
pretrain_dict={k:v for k,v in state_dict.items()if k in model_dic}
model_dict.update(pretrain_dict)
model.load_state_dict(model_dict)
model.eval()
    
    

In [ ]:
tokenizer=GPT2Tokenizer.from_pretrained("gpt2")
request= torch.tensor([tokenizer.encode("The planet earth is a beautiful")])
def generate(context,ntokens=550):
    start=time.time()
    for _ in range (ntokens):
        out=model (context)
        logits=out[:-1:]# take all baches , laast  token in the sequence and all vocabulary logits
        indices_to_remove=logits<torch.topk(logits,10)[...,-1,None] #take all batches the less of topk  and use it to compare
        logits[indices_to_remove]=-np.inf
        #from top k samples 1 answer
        next_tok=torch.mutinomial(func.softmax(logits,dim=-1),num_samples=1).squeez(1)
        context=torch.cat([context,next_tok.unsqueeze(-1)],dim=-1)
    end=time.time()
    inference_time=end-start
    return inference_time, context
        

In [ ]:
out,inference_time=generate(content,ntoken=40)
decoded_output=tokenizer.decode(out[0])
print(f"inference Time:{inference_time:.4f} seconds")
print(f"generated output:{decoded_output}")